*Your Name*

*Collaborator's Names*

# Improving Performance with NumPy

NumPy, Pandas, and other packages in the scientific Python ecosystem provide a huge variety of tools that make common research tasks simpler. Generally, its best to use these tools as they have been specially optimized; however, there can be reasons to prefer code that takes slightly longer to run. Sometimes performant code is hard to read. Other times, developers may have had other goals in mind, and don't prioritize performance. **Always, always test your code changes to ensure you're actually seeing an improvement** (and that your results are correct).

The code used in this workbook is modified from [High Performance Python by Gorelick & Ozsvald](https://github.com/mynameisfiber/high_performance_python_2e/tree/master/06_matrix).

---
## Refactoring Lists to NumPy Arrays



Last week we learned that NumPy arrays can take advantage of vectorization to efficiently perform mathematical operations on lost of (homogenous) data. The goal of this section is to provide experience converting an existing list-based code to use NumPy. It also introduces NumPy array slicing - another way NumPy leverages vectorization.

The code below simulates a simple 1D diffusion problem. Entries in a list function as cells in a grid,
where each cell's value represents the concentration of a fluid at a point in space. Over time, the concentration in each cell will change as the fluid redistributes. **You will be incrementally refactoring this list-based code to use NumPy.**

A small region near the center of the grid is given a higher concentration than its surroundings:

![Graph of the initial conditions for our sample 1D diffusion problem.](diffusion_ICs.png "Diffusion Initial Conditions")

After 500 iterations (timesteps) with the default parameters, the concentration smooths out:

![Graph of the diffusion problem after 500 timesteps.](diffusion_updated.png "Diffusion After 500 Timesteps")

### Algorithm Description

Each entry in the list `grid` corresponds to some coordinate along the $x$ axis, such that the $i$ th cell of `grid` corresponds to position $x_{i}$.
The concentration C at position $x_i$ --- or $C(x_i)$ ---  is then stored as `grid[i]`. 

Each time we update the grid --- taking a timestep `dt` --- we iterate over all the cells in the grid. To update the value of each cell,
we apply a "stencil" which combines information from neighboring cells in a weighted manner.

For cell $i$, the stencil uses data from cells $i-1$ and $i+1$ to approximate the second derivative of the concentration $C$:

$\frac{dC}{dt} = D \cdot \frac{d^2}{dx^2} C(x) \approx D \cdot \left( C(x_{i-1}) + C(x_{i+1}) - 2 C(x) \right)$

where $D$ is the diffusion coefficient.

Below is an example grid. The stencil is shown in blue. The red cells are special cells called "ghost zones." These cells exist so that the stencil can still be applied to the edges of the grid. Information in ghost zones is usually ignored when analyzing the results of simulations like this 1D diffusion problem, but are **very** important to the correct function of the diffusion algorithm.

![Diagram the stencil-based grid update and ghost zones](stencil.png)



In [1]:
import numpy as np

In [2]:
def run_simulation(grid_size = 600, 
                   dt = 0.1,
                   num_timesteps = 500,
                   diffusion_coeff = 1.0):
    """
    Simulate the diffusion of a fluid in 1D using lists.
    
    grid_size: the number of cells in our grid.
    dt: time change for each iteration
    num_timesteps: how many time steps we allow the fluid to diffuse.
    diffusion_coeff: diffusivity of the fluid; higher is more diffusive.

    Returns the state of the grid after num_timesteps
    """

    # Construct 1D grid
    # Add an additional cell at each end to handle grid boundaries
    # grid[0] and grid[grid_size+1] are these boundary cells
    grid = [0.01] * (grid_size + 2)

    # Set the initial conditions
    # A small region (~10% of total length)
    # near the middle has high concentration
    start_index = int(grid_size * 0.4)
    end_index = int(grid_size * 0.5)
    for i in range(start_index+1, end_index+1):
        grid[i] = 0.02

    # Evolve the grid
    for t in range(num_timesteps):

        # create a new grid to store updates
        new_grid = [0.0] * (grid_size + 2)

        # update main grid
        for i in range(1, grid_size+1):
            stencil = grid[i+1] + grid[i-1] - 2*grid[i]
            new_grid[i] = grid[i] + diffusion_coeff * stencil * dt

        # update boundary cells
        new_grid[0] = new_grid[1]
        new_grid[grid_size+1] = new_grid[grid_size]

        # swap new_grid and grid
        grid, new_grid = new_grid, grid

    return grid


---
### Exercises

1. Time `run_simulation` to establish a baseline. You can use the default arguments.

*Record timing here*

2. Use `lprun` to profile the existing list-based version of `run_simulation`. Even though all of our code is in a single function, you will still need to specify the `-f` argument. The following questions will require the profiling output to answer.

3. Which lines take the largest fraction of the overall runtime? What purpose do these lines serve?

*Answer here*

4. Which lines take the most amount of time *per hit*? Is there a common theme between these lines? Think about memory --- allocation, access, data movement, etc.

*Answer here*

5. Defining a new function, convert `grid` and `new_grid` to NumPy arrays (recall the [array creation](https://numpy.org/doc/1.25/reference/routines.array-creation.html#numerical-ranges) routines). Check the results against the figures shown above and time your refactor. Is this version of the code slower or faster? Why do you think that is?

*Answer here*

6. Refactor the diffusion code a second time. This time, eliminate the `for` loops by using [array slicing](https://numpy.org/doc/stable/user/absolute_beginners.html#indexing-and-slicing). This will enable us to use vectorization.

Hint: the loop
```python
for i in range(1, grid_size+1):
    grid[i]
```
will become
```python
grid[1:-1]
```
Drawing a diagram may help you, especially when handling the ghost zones!

7. Time the slice-based code for 500 timesteps. How does the speed of this change compare to both our previous versions? Is this what you expect?

*Answer here*

8. There is one last easy refactor we can do (if you are familiar with this style of algorithm, you may have already spotted it). Profile the vectorized 1D diffusion code and consider the lines with multiple hits. We have been able to reduce the time spent on some of these lines thanks to vectorization. Another strategy is simply to reduce the number of times a line is executed! Which line can benefit from this kind of optimization without breaking the algorithm?

*Answer here*

9. Refactor the code a final time to implement the optimization identified above. Compared to the *original* version, what is the final factor of speed-up achieved?

*Answer here*